<a href="https://colab.research.google.com/github/vangogh1803/face_rec_final/blob/main/face_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/face_proj"


Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q datasets
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json


cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!pip install facenet-pytorch scikit-image tqdm opencv-python-headless


In [ ]:
!kaggle datasets download -d hearfool/vggface2
!unzip vggface2.zip -d /content/vggface2

Streaming output truncated to the last 5000 lines.
  inflating: /content/vggface2/val/n001146/0284_01.jpg  
  inflating: /content/vggface2/val/n001146/0285_01.jpg  
  inflating: /content/vggface2/val/n001146/0286_01.jpg  
  inflating: /content/vggface2/val/n001146/0287_02.jpg  
  inflating: /content/vggface2/val/n001146/0288_01.jpg  
  inflating: /content/vggface2/val/n001146/0289_01.jpg  
  inflating: /content/vggface2/val/n001146/0290_01.jpg  
  inflating: /content/vggface2/val/n001146/0291_01.jpg  
  inflating: /content/vggface2/val/n001146/0292_01.jpg  
  inflating: /content/vggface2/val/n001146/0293_01.jpg  
  inflating: /content/vggface2/val/n001146/0294_02.jpg  
  inflating: /content/vggface2/val/n001146/0295_02.jpg  
  inflating: /content/vggface2/val/n001146/0296_01.jpg  
  inflating: /content/vggface2/val/n001146/0297_03.jpg  
  inflating: /content/vggface2/val/n001146/0298_01.jpg  
  inflating: /content/vggface2/val/n001146/0299_01.jpg  
  inflating: /content/vggface2/val/n0

In [ ]:
import os, random, shutil, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from sklearn.metrics import roc_curve
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

# ── CONFIG ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ALIGNED_DATA = "/content/vggface2/train"
SUBSET_PATH  = "/content/vgg_subset"

N_IDS         = 1900
N_IMGS        = 20
BATCH_SIZE    = 128
EPOCHS        = 90
PATIENCE      = 15
UNFREEZE_EP   = 5
WARMUP_EPOCHS = 3
K_SUBCENTERS  = 3
SEED          = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── 1. CREATE SUBSET ───────────────────────────────────
if not os.path.exists(SUBSET_PATH):
    ids = [d for d in os.listdir(ALIGNED_DATA)
           if os.path.isdir(os.path.join(ALIGNED_DATA, d))]

    selected = random.sample(ids, min(N_IDS, len(ids)))

    valid_count = 0

    for identity in selected:
        src = os.path.join(ALIGNED_DATA, identity)
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg','.png','.jpeg'))]

        if len(imgs) < 5:   # 🔥 FIXED
            continue

        imgs = random.sample(imgs, min(N_IMGS, len(imgs)))

        dst = os.path.join(SUBSET_PATH, identity)
        os.makedirs(dst, exist_ok=True)

        for f in imgs:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))

        valid_count += 1

    print(f"✅ Subset created with {valid_count} identities")
else:
    print("Subset exists")

# ── 2. LOAD DATA ───────────────────────────────────────
def get_samples(root):
    out = []
    for identity in os.listdir(root):
        p = os.path.join(root, identity)
        if not os.path.isdir(p): continue

        imgs = [os.path.join(p, f) for f in os.listdir(p)
                if f.lower().endswith(('.jpg','.png','.jpeg'))]

        if len(imgs) >= 5:   # 🔥 FIXED
            out.append((identity, imgs))

    return out

all_ids = get_samples(SUBSET_PATH)
random.shuffle(all_ids)

n_classes = len(all_ids)
id_to_idx = {identity: i for i, (identity, _) in enumerate(all_ids)}

train_samples = []
gallery = defaultdict(list)

for identity, imgs in all_ids:
    random.shuffle(imgs)

    n_probe = max(10, int(0.45 * len(imgs)))

    for p in imgs[n_probe:]:
        train_samples.append((p, id_to_idx[identity]))

    for p in imgs[:n_probe]:
        gallery[identity].append(p)

print(f"Classes: {n_classes}, Train imgs: {len(train_samples)}")

# ── 3. TRANSFORMS ──────────────────────────────────────
train_tfm = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(112),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.3,0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tfm = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(112),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── 4. DATASETS ────────────────────────────────────────
class TrainDataset(Dataset):
    def __init__(self, samples, tfm):
        self.samples = samples
        self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        try:
            img = Image.open(path).convert('RGB')
            img = self.tfm(img)
            return img, label
        except:
            return None, label

class GalleryDataset(Dataset):
    def __init__(self, gallery_dict, tfm):
        self.samples = []
        for identity, paths in gallery_dict.items():
            for p in paths:
                self.samples.append((p, identity))
        self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, identity = self.samples[i]
        try:
            img = Image.open(path).convert('RGB')
            img = self.tfm(img)
            return img, identity
        except:
            return None, identity

# ── COLLATE ────────────────────────────────────────────
def train_collate_fn(batch):
    batch = [b for b in batch if b[0] is not None]
    if len(batch) == 0:
        return None, None
    imgs, labels = zip(*batch)
    return torch.stack(imgs), torch.tensor(labels)

def gallery_collate_fn(batch):
    batch = [b for b in batch if b[0] is not None]
    if len(batch) == 0:
        return None, None
    imgs, labels = zip(*batch)
    return torch.stack(imgs), list(labels)

# ── DATALOADER ─────────────────────────────────────────
train_loader = DataLoader(
    TrainDataset(train_samples, train_tfm),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=train_collate_fn
)

gallery_loader = DataLoader(
    GalleryDataset(gallery, val_tfm),
    batch_size=128,
    num_workers=2,
    collate_fn=gallery_collate_fn
)

# ── MODEL ──────────────────────────────────────────────
class SubcenterArcFace(nn.Module):
    def __init__(self, in_feat, n_classes, K=3, s=48.0, m=0.4):
        super().__init__()
        self.s = s
        self.m = m
        self.K = K
        self.weight = nn.Parameter(torch.FloatTensor(n_classes * K, in_feat))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        cosine_all = F.linear(F.normalize(x), F.normalize(self.weight))
        cosine = cosine_all.view(cosine_all.size(0), -1, self.K).max(dim=2).values
        cosine = cosine.clamp(-1, 1)

        theta = torch.acos(cosine)
        phi = torch.cos(theta + self.m)

        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1,1), 1.0)
        return (one_hot * phi + (1 - one_hot) * cosine) * self.s

backbone = InceptionResnetV1(pretrained='vggface2').to(device)

for p in backbone.parameters():
    p.requires_grad = False

head = SubcenterArcFace(512, n_classes, K_SUBCENTERS).to(device)

optimizer = optim.AdamW([
    {'params': head.parameters(), 'lr': 3e-4}
], weight_decay=5e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2
)

# ── EVALUATE ───────────────────────────────────────────
@torch.no_grad()
def evaluate(backbone, loader):
    backbone.eval()
    feats, labels = [], []

    for imgs, lbls in loader:
        if imgs is None:
            continue

        imgs = imgs.to(device)
        f = F.normalize(backbone(imgs), dim=1)
        feats.append(f.cpu())
        labels.extend(lbls)

    if len(feats) == 0:
        return 1.0

    feats = torch.cat(feats)

    id_to_idxs = defaultdict(list)
    for i, lbl in enumerate(labels):
        id_to_idxs[lbl].append(i)

    pairs = []
    for v in id_to_idxs.values():
        pairs += [(i,j,1) for i,j in itertools.combinations(v,2)]

    pairs = pairs[:60000]

    neg = []
    keys = list(id_to_idxs.keys())
    while len(neg) < len(pairs):
        a,b = random.sample(keys,2)
        i = random.choice(id_to_idxs[a])
        j = random.choice(id_to_idxs[b])
        neg.append((i,j,0))

    pairs += neg

    i_list = [p[0] for p in pairs]
    j_list = [p[1] for p in pairs]
    targets = np.array([p[2] for p in pairs])

    scores = torch.sum(feats[i_list] * feats[j_list], dim=1).numpy()

    fpr, tpr, _ = roc_curve(targets, scores)
    eer = fpr[np.nanargmin(np.abs(fpr - (1 - tpr)))]

    return eer

# ── TRAIN ──────────────────────────────────────────────
best_eer = 1.0
no_improve = 0

for epoch in range(EPOCHS):

    if epoch < WARMUP_EPOCHS:
        for g in optimizer.param_groups:
            g['lr'] = 1e-4

    if epoch == UNFREEZE_EP:
        for p in backbone.parameters():
            p.requires_grad = True

        optimizer.add_param_group({
            'params': backbone.parameters(),
            'lr': 6e-4
        })

    backbone.train(); head.train()
    total_loss = 0

    for imgs, lbls in tqdm(train_loader):

        if imgs is None:
            continue

        imgs = imgs.to(device)
        lbls = lbls.to(device)

        optimizer.zero_grad()

        emb = backbone(imgs)
        emb = F.dropout(emb, p=0.2, training=True)
        emb = F.normalize(emb, dim=1)

        logits = head(emb, lbls)
        loss = F.cross_entropy(logits / 2, lbls, label_smoothing=0.1)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(backbone.parameters()) + list(head.parameters()), 5.0
        )

        optimizer.step()
        total_loss += loss.item()

    scheduler.step()

    eer = evaluate(backbone, gallery_loader)

    print(f"Epoch {epoch+1} | Loss {total_loss:.2f} | EER {eer*100:.2f}%")

    if eer < best_eer:
        best_eer = eer
        no_improve = 0
        torch.save(backbone.state_dict(), "best_vgg.pt")
        print("✅ Saved model")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping")
            break

print(f"\n🔥 FINAL BEST EER: {best_eer*100:.2f}%")

✅ Subset created with 480 identities
Classes: 480, Train imgs: 4800


100%|██████████| 38/38 [00:12<00:00,  2.94it/s]


Epoch 1 | Loss 566.78 | EER 21.06%
✅ Saved model


100%|██████████| 38/38 [00:12<00:00,  2.94it/s]


Epoch 2 | Loss 557.59 | EER 21.80%


...

Epoch 26 | Loss 48.88 | EER 17.74%


100%|██████████| 38/38 [00:14<00:00,  2.54it/s]


Epoch 27 | Loss 49.14 | EER 18.08%


100%|██████████| 38/38 [00:14<00:00,  2.54it/s]


Epoch 28 | Loss 48.85 | EER 17.26%
Early stopping

🔥 FINAL BEST EER: 13.56%


In [ ]:
rm -rf /content/vgg_subset

In [ ]:
import gc
import torch

# 1. Delete your model and data variables if they exist
if 'backbone' in locals(): del backbone
if 'head' in locals(): del head
if 'train_loader' in locals(): del train_loader

# 2. Push Python to clear standard RAM
gc.collect()

# 3. Force PyTorch to dump its cache back to the GPU
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
"""
Face Verification — InceptionResNetV1 (VGGFace2) + SubcenterArcFace
====================================================================
Target  : EER < 3% on VGGFace2 subset (1900 ids, ~20 imgs each)
Backbone: facenet-pytorch InceptionResNetV1, pretrained='vggface2'
Loss    : SubcenterArcFace  K=1 (i.e. standard ArcFace) for small sets
Eval    : Deterministic pairs + flip-TTA (built once, reused every epoch)

Key fixes over the 13.5% version
─────────────────────────────────
1. Backbone trains from epoch 0 with a low differential LR (1e-5).
   Freezing for 5 epochs wastes the pretrained features and forces the
   head to learn on a fixed representation that it will never see again.

2. Gallery split fixed: 20% gallery (was 45%). With only 20 imgs/id,
   45% gallery = 9 gallery + 11 train, which is fine for eval but hurts
   training badly. 20% = 4 gallery + 16 train is a better trade-off.

3. Deterministic eval pairs (seeded, built once). Random re-sampling
   caused EER to jump ±4% between epochs — best checkpoint was noise.

4. Flip-TTA in eval: embed(img) + embed(hflip(img)), then L2-norm.
   Free ~1-2% EER improvement on face data.

5. Removed `logits / 2` division — this was halving the effective
   scale from 48 to 24 and making the loss landscape very flat.

6. K=1 subcenters (standard ArcFace) for this dataset size.
   K=3 with only ~16 train imgs/class means each subcenter sees ~5
   samples — not enough to form meaningful intra-class clusters.
   Use K=1 now; if you scale to >500 imgs/class, bump to K=3.

7. Warmup-hold-cosine LR schedule (epoch-relative, resume-safe).
   CosineAnnealingWarmRestarts was resetting LR after unfreeze and
   spiking the backbone's gradients right when it started training.

8. Label smoothing moved to nn.CrossEntropyLoss (not the logit-halving
   hack). ε=0.05 (was 0.1) — less smoothing is better with ArcFace
   since the margin already acts as a regulariser.

9. AMP + gradient accumulation for OOM safety on 15GB GPUs.

10. torch.compile() opt-in for ~15% extra speed on PyTorch ≥ 2.0.
"""

import os, random, shutil, itertools, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
from sklearn.metrics import roc_curve
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

# ── CONFIG ───────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ALIGNED_DATA = "/content/vggface2/train"
SUBSET_PATH  = "/content/vgg_subset"

N_IDS        = 1900
N_IMGS       = 20
BATCH_SIZE   = 64           # effective 128 with ACC_STEPS=2
ACC_STEPS    = 2
EPOCHS       = 90
PATIENCE     = 20
IMG_SIZE     = 160          # InceptionResnetV1 native size
SEED         = 42

NUM_WORKERS  = 4
EVAL_START   = 2            # first eval at epoch 3 (0-indexed: 2)
EVAL_EVERY   = 2
USE_AMP      = True
USE_COMPILE  = False

# SubcenterArcFace
K_SUB        = 1            # 1 = standard ArcFace; bump to 3 if >200 imgs/id
ARC_S        = 48.0
ARC_M        = 0.35         # slightly softer than 0.4 for small sets

# LR schedule (warmup → hold → cosine decay)
BB_LR_PEAK   = 1e-5         # backbone — pretrained, keep conservative
HEAD_LR_PEAK = 3e-4
BB_LR_INIT   = 1e-7
HEAD_LR_INIT = 1e-7
WARMUP_EP    = 3
HOLD_EP      = 10

LABEL_SMOOTH = 0.05         # small; ArcFace margin already regularises

RESUME_FROM  = None         # e.g. "best_vgg.pt" to continue training

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── 1. SUBSET ────────────────────────────────────────────────────────
if not os.path.exists(SUBSET_PATH):
    ids = [d for d in os.listdir(ALIGNED_DATA)
           if os.path.isdir(os.path.join(ALIGNED_DATA, d))]
    selected = random.sample(ids, min(N_IDS, len(ids)))
    count = 0
    for ident in selected:
        src  = os.path.join(ALIGNED_DATA, ident)
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(imgs) < 5:
            continue
        imgs = random.sample(imgs, min(N_IMGS, len(imgs)))
        dst  = os.path.join(SUBSET_PATH, ident)
        os.makedirs(dst, exist_ok=True)
        for f in imgs:
            shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        count += 1
    print(f"✅ Subset: {count} identities")
else:
    print("Subset already exists.")

# ── 2. SPLIT ─────────────────────────────────────────────────────────
def get_samples(root):
    out = []
    for ident in os.listdir(root):
        p = os.path.join(root, ident)
        if not os.path.isdir(p): continue
        imgs = [os.path.join(p, f) for f in os.listdir(p)
                if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if len(imgs) >= 5:
            out.append((ident, imgs))
    return out

all_ids   = get_samples(SUBSET_PATH)
random.shuffle(all_ids)
n_classes = len(all_ids)
id_to_idx = {ident: i for i, (ident, _) in enumerate(all_ids)}

train_samples = []
gallery       = defaultdict(list)

for ident, imgs in all_ids:
    random.shuffle(imgs)
    # 20% gallery — keeps max images for training
    n_g = max(2, int(0.20 * len(imgs)))
    for p in imgs[n_g:]:  train_samples.append((p, id_to_idx[ident]))
    for p in imgs[:n_g]:  gallery[ident].append(p)

print(f"Classes: {n_classes} | Train: {len(train_samples)} | "
      f"Gallery: {sum(len(v) for v in gallery.values())}")

# ── 3. TRANSFORMS ────────────────────────────────────────────────────
train_tfm = transforms.Compose([
    transforms.Resize(180),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tfm = transforms.Compose([
    transforms.Resize(180),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── 4. DATASETS ──────────────────────────────────────────────────────
class TrainDataset(Dataset):
    def __init__(self, samples, tfm):
        self.samples = samples; self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        try:    return self.tfm(Image.open(path).convert('RGB')), label
        except: return None, label

class GalleryDataset(Dataset):
    def __init__(self, gd, tfm):
        self.samples = [(p, ident) for ident, ps in gd.items() for p in ps]
        self.tfm = tfm

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, ident = self.samples[i]
        try:
            img = Image.open(path).convert('RGB')
            return (self.tfm(img),
                    self.tfm(transforms.functional.hflip(img)),
                    ident)
        except: return None, None, ident

def train_collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None, None
    imgs, lbls = zip(*batch)
    return torch.stack(imgs), torch.tensor(lbls)

def gallery_collate(batch):
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None, None, None
    i1, i2, lbls = zip(*batch)
    return torch.stack(i1), torch.stack(i2), list(lbls)

train_loader = DataLoader(
    TrainDataset(train_samples, train_tfm),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=train_collate,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

gallery_loader = DataLoader(
    GalleryDataset(gallery, val_tfm),
    batch_size=128, num_workers=NUM_WORKERS,
    collate_fn=gallery_collate,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

# ── 5. DETERMINISTIC EVAL PAIRS ──────────────────────────────────────
def build_eval_pairs(gallery_dict, max_pairs=20000, seed=0):
    rng     = random.Random(seed)
    ordered = [(p, ident) for ident, ps in gallery_dict.items() for p in ps]
    id2idx  = defaultdict(list)
    for idx, (_, ident) in enumerate(ordered):
        id2idx[ident].append(idx)

    pos = []
    for v in id2idx.values():
        pos += list(itertools.combinations(v, 2))
    rng.shuffle(pos)
    pos = pos[:max_pairs]

    keys = list(id2idx.keys())
    neg  = []
    while len(neg) < len(pos):
        a, b = rng.sample(keys, 2)
        neg.append((rng.choice(id2idx[a]), rng.choice(id2idx[b])))

    I = torch.tensor([p[0] for p in pos + neg])
    J = torch.tensor([p[1] for p in pos + neg])
    T = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int32)
    return I, J, T

print("Building deterministic eval pairs …")
EVAL_I, EVAL_J, EVAL_T = build_eval_pairs(gallery, max_pairs=20000)
print(f"Eval: {len(EVAL_T)//2} pos + {len(EVAL_T)//2} neg")

# ── 6. MODEL ─────────────────────────────────────────────────────────
class SubcenterArcFace(nn.Module):
    """
    K=1  →  standard ArcFace  (recommended for <200 imgs/class)
    K>1  →  subcenter variant  (for noisier / larger datasets)
    """
    def __init__(self, in_feat, n_classes, K=1, s=48.0, m=0.35):
        super().__init__()
        self.s = s; self.m = m; self.K = K
        self.weight = nn.Parameter(torch.FloatTensor(n_classes * K, in_feat))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        cos_all = F.linear(F.normalize(x), F.normalize(self.weight))  # (B, C*K)
        if self.K > 1:
            cos = cos_all.view(x.size(0), -1, self.K).max(dim=2).values
        else:
            cos = cos_all
        cos = cos.clamp(-1 + 1e-7, 1 - 1e-7)

        theta   = torch.acos(cos)
        phi     = torch.cos(theta + self.m)
        one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.)
        # No logit division — use full scale s
        return (one_hot * phi + (1 - one_hot) * cos) * self.s


# InceptionResnetV1 pretrained on VGGFace2 — already a face expert
backbone = InceptionResnetV1(pretrained='vggface2').to(device)
head     = SubcenterArcFace(512, n_classes, K=K_SUB,
                            s=ARC_S, m=ARC_M).to(device)

if USE_COMPILE and hasattr(torch, 'compile'):
    backbone = torch.compile(backbone)
    print("  ✅ torch.compile() applied.")

# Differential LR: backbone gets 30× lower LR than head
optimizer = optim.AdamW([
    {'params': backbone.parameters(), 'lr': BB_LR_INIT},
    {'params': head.parameters(),     'lr': HEAD_LR_INIT},
], weight_decay=5e-4)

ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# ── 7. LR SCHEDULE (epoch-relative, resume-safe) ─────────────────────
def set_lr(epoch, start_epoch=0):
    """
    Warmup → hold → cosine, measured relative to start_epoch so that
    resuming mid-run doesn't reset the schedule back to warmup.
    """
    rel      = epoch - start_epoch
    remain   = EPOCHS - start_epoch
    decay_ep = remain - WARMUP_EP - HOLD_EP

    if rel < WARMUP_EP:
        f       = (rel + 1) / WARMUP_EP
        bb_lr   = BB_LR_INIT   + f * (BB_LR_PEAK   - BB_LR_INIT)
        head_lr = HEAD_LR_INIT + f * (HEAD_LR_PEAK - HEAD_LR_INIT)
    elif rel < WARMUP_EP + HOLD_EP:
        bb_lr, head_lr = BB_LR_PEAK, HEAD_LR_PEAK
    else:
        t       = (rel - WARMUP_EP - HOLD_EP) / max(1, decay_ep)
        factor  = max(0.01, 0.5 * (1 + math.cos(math.pi * t)))
        bb_lr   = BB_LR_PEAK   * factor
        head_lr = HEAD_LR_PEAK * factor

    optimizer.param_groups[0]['lr'] = bb_lr
    optimizer.param_groups[1]['lr'] = head_lr

# ── 8. RESUME ────────────────────────────────────────────────────────
start_epoch = 0
best_eer    = 1.0
if RESUME_FROM and os.path.isfile(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device)
    backbone.load_state_dict(ckpt['backbone'])
    head.load_state_dict(ckpt['head'])
    start_epoch = ckpt.get('epoch', 0)
    best_eer    = ckpt.get('eer', 1.0)
    print(f"  ✅ Resumed from epoch {start_epoch}  (EER {best_eer*100:.2f}%)")
else:
    print("  Starting from scratch.")

# ── 9. EVALUATE ──────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(backbone):
    backbone.eval()
    feats = []
    for batch in gallery_loader:
        if batch[0] is None: continue
        i1, i2, _ = batch
        i1, i2 = i1.to(device), i2.to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            f1 = F.normalize(backbone(i1), dim=1)
            f2 = F.normalize(backbone(i2), dim=1)
        # Flip-TTA: fuse normal + flipped embedding
        feats.append(F.normalize(f1 + f2, dim=1).cpu())
    if not feats: return 1.0
    feats  = torch.cat(feats)
    scores = (feats[EVAL_I] * feats[EVAL_J]).sum(1).numpy()
    fpr, tpr, _ = roc_curve(EVAL_T, scores)
    return float(fpr[np.nanargmin(np.abs(fpr - (1 - tpr)))])

# ── 10. TRAIN ────────────────────────────────────────────────────────
no_improve = 0

for epoch in range(start_epoch, EPOCHS):
    set_lr(epoch, start_epoch)

    backbone.train(); head.train()
    epoch_loss = 0.0
    n_batches  = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Ep {epoch+1:03d}", leave=False)
    for i, (imgs, lbls) in enumerate(pbar):
        if imgs is None: continue

        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            emb    = F.normalize(backbone(imgs), dim=1)
            logits = head(emb, lbls)
            loss   = ce_loss(logits, lbls) / ACC_STEPS

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * ACC_STEPS
        n_batches  += 1

        if (i + 1) % ACC_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                list(backbone.parameters()) + list(head.parameters()), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item()*ACC_STEPS:.3f}")

    do_eval = (epoch >= EVAL_START) and (
        (epoch - EVAL_START) % EVAL_EVERY == 0 or epoch == EPOCHS - 1)

    eer     = evaluate(backbone) if do_eval else float('nan')
    bb_lr   = optimizer.param_groups[0]['lr']
    eer_str = f"{eer*100:5.2f}%" if not np.isnan(eer) else "  --  "
    avg     = epoch_loss / max(n_batches, 1)

    print(f"Epoch {epoch+1:3d} | Loss {avg:6.3f} | EER {eer_str} | "
          f"LR bb={bb_lr:.1e} hd={optimizer.param_groups[1]['lr']:.1e}")

    if not np.isnan(eer):
        if eer < best_eer:
            best_eer   = eer
            no_improve = 0
            torch.save({
                'epoch'   : epoch + 1,
                'backbone': backbone.state_dict(),
                'head'    : head.state_dict(),
                'eer'     : best_eer,
            }, "best_vgg.pt")
            print(f"  ✅ Saved  (EER {best_eer*100:.2f}%)")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print("Early stopping.")
                break

print(f"\n🔥 FINAL BEST EER: {best_eer*100:.2f}%")

✅ Subset: 480 identities
Classes: 480 | Train: 7680 | Gallery: 1920
Building deterministic eval pairs …
Eval: 2880 pos + 2880 neg
  Starting from scratch.


Epoch   1 | Loss 23.172 | EER   --   | LR bb=3.4e-06 hd=1.0e-04


Epoch   2 | Loss 21.029 | EER   --   | LR bb=6.7e-06 hd=2.0e-04


Epoch   3 | Loss 18.201 | EER  3.85% | LR bb=1.0e-05 hd=3.0e-04
  ✅ Saved  (EER 3.85%)


Epoch   4 | Loss 15.756 | EER   --   | LR bb=1.0e-05 hd=3.0e-04


...

Epoch  61 | Loss  1.197 | EER  3.06% | LR bb=3.3e-06 hd=9.9e-05


Epoch  62 | Loss  1.191 | EER   --   | LR bb=3.1e-06 hd=9.3e-05


Epoch  63 | Loss  1.188 | EER  3.02% | LR bb=2.9e-06 hd=8.8e-05


Epoch  64 | Loss  1.195 | EER   --   | LR bb=2.7e-06 hd=8.2e-05


Epoch  65 | Loss  1.179 | EER  3.12% | LR bb=2.6e-06 hd=7.7e-05
Early stopping.

🔥 FINAL BEST EER: 2.64%


In [ ]:
import torchaudio

import torchaudio

dataset = torchaudio.datasets.LIBRISPEECH(
    root="/content",
    url="train-clean-100",
    download=True
)

print("✅ LibriSpeech downloaded")

In [ ]:
import os, shutil, random

SRC = "/content/LibriSpeech/train-clean-100"
DST = "/content/audio_dataset"

os.makedirs(DST, exist_ok=True)

N_SPEAKERS = 20   # 🔥 reduce if slow
N_SAMPLES = 25    # per speaker

speakers = os.listdir(SRC)
selected = random.sample(speakers, min(N_SPEAKERS, len(speakers)))

for spk in selected:
    spk_src = os.path.join(SRC, spk)
    spk_dst = os.path.join(DST, spk)
    os.makedirs(spk_dst, exist_ok=True)

    all_files = []

    for chapter in os.listdir(spk_src):
        ch_path = os.path.join(spk_src, chapter)
        for f in os.listdir(ch_path):
            if f.endswith(".flac"):
                all_files.append(os.path.join(ch_path, f))

    chosen = random.sample(all_files, min(N_SAMPLES, len(all_files)))

    for f in chosen:
        shutil.copy(f, spk_dst)

print("✅ Audio subset ready!")

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve
import torch.nn.functional as F

# Load audio embeddings
audio = torch.load("/content/audio_results.pt")

audio_emb = audio["embeddings"]
labels = np.array(audio["labels"])

# Normalize
audio_emb = F.normalize(audio_emb, dim=1)

# -------------------------------
# CREATE PAIRS
# -------------------------------
import random

pairs = []
pair_labels = []

# group indices by label
label_dict = {}
for idx, lbl in enumerate(labels):
    label_dict.setdefault(lbl, []).append(idx)

# POSITIVE pairs
for lbl, indices in label_dict.items():
    for i in range(len(indices)-1):
        sim = torch.dot(audio_emb[indices[i]], audio_emb[indices[i+1]]).item()
        pairs.append(sim)
        pair_labels.append(1)

# NEGATIVE pairs
all_labels = list(label_dict.keys())

for _ in range(len(pairs)):
    a, b = random.sample(all_labels, 2)
    i = random.choice(label_dict[a])
    j = random.choice(label_dict[b])

    sim = torch.dot(audio_emb[i], audio_emb[j]).item()
    pairs.append(sim)
    pair_labels.append(0)
print("✅ Audio pairs:", len(pairs))


In [ ]:
import torchaudio.transforms as T

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
)

In [ ]:
import torchaudio.transforms as T

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
)

In [ ]:
import torchaudio

bundle = torchaudio.pipelines.WAV2VEC2_BASE
wav2vec = bundle.get_model().to(device)
wav2vec.eval()

print("✅ wav2vec loaded")

In [ ]:
class SimpleAudioModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(40, 128)
        self.fc2 = nn.Linear(128, 128)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (B, time)
        mfcc = mfcc_transform(x)   # (B, 40, T)
        mfcc = mfcc.mean(dim=2)    # average over time

        emb = torch.relu(self.fc1(mfcc))
        emb = F.normalize(self.fc2(emb), dim=1)

        logits = self.classifier(emb)

        return emb, logits

In [ ]:
import torchaudio.transforms as T

mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40
).to(device)
model = SimpleAudioModel(num_classes=len(dataset.label_map)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for waveform, label in loader:
        waveform = waveform.to(device)
        label = label.to(device)

        emb, logits = model(waveform)   # ✅ FIXED

        loss = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.3f}")

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve
import torch.nn.functional as F

audio = torch.load("audio_results.pt")

audio_emb = F.normalize(audio["embeddings"], dim=1)
labels = np.array(audio["labels"])

pairs = []
pair_labels = []

N = len(audio_emb)
MAX_PAIRS = 10000

count = 0

for i in range(N):
    for j in range(i+1, N):
        if count >= MAX_PAIRS:
            break

        sim = torch.dot(audio_emb[i], audio_emb[j]).item()
        pairs.append(sim)

        if labels[i] == labels[j]:
            pair_labels.append(1)
        else:
            pair_labels.append(0)

        count += 1
    if count >= MAX_PAIRS:
        break

pairs = np.array(pairs)
pair_labels = np.array(pair_labels)

# ROC
fpr, tpr, thresholds = roc_curve(pair_labels, pairs)
frr = 1 - tpr

eer_idx = np.nanargmin(np.abs(fpr - frr))
eer = fpr[eer_idx]

print(f"🎧 Audio EER: {eer*100:.2f}%")

# FAR / FRR
th = thresholds[eer_idx]

far = np.mean((pairs >= th) & (pair_labels == 0))
frr_val = np.mean((pairs < th) & (pair_labels == 1))

print(f"FAR: {far*100:.2f}%")
print(f"FRR: {frr_val*100:.2f}%")